# 🔬 Quantum Random Walk with Decoherence Simulation

This notebook explores how a quantum particle spreads across a 1D lattice using the **discrete-time quantum walk** model, and how environmental noise (**decoherence**) gradually destroys quantum behaviour — bridging Quantum Computing and Quantum Field Theory concepts.

## Key Concepts Covered
- **Coin Operator**: Unitary matrix that creates superposition (analogous to a quantum gate)
- **Shift Operator**: Moves the particle based on its coin state
- **Decoherence**: Interaction with the environment, causing quantum→classical transition
- **Quadratic Speedup**: Quantum walk spreads as σ ∝ t, classical spreads as σ ∝ √t

In [ ]:
import sys
sys.path.append('../src')
import numpy as np
import matplotlib.pyplot as plt
from quantum_walk import (
    quantum_walk_1d, classical_random_walk,
    hadamard_coin, grover_coin, plot_comparison
)
print('Imports successful ✅')

## 1. Hadamard Coin — The Simplest Quantum Coin
The Hadamard gate puts the coin in equal superposition of ↑ and ↓ states.

In [ ]:
H = hadamard_coin()
print('Hadamard Coin Matrix:')
print(np.round(H, 3))
print(f'\nUnitary check (H†H = I): {np.allclose(H.conj().T @ H, np.eye(2))}')

## 2. Single Walk: Quantum vs Classical
Run 100 steps and compare the probability distributions.

In [ ]:
STEPS = 100
positions = np.arange(-STEPS, STEPS + 1)

q_prob = quantum_walk_1d(STEPS, hadamard_coin(), decoherence=0.0)
c_prob = classical_random_walk(STEPS)

q_std = np.sqrt(np.sum(q_prob * positions**2))
c_std = np.sqrt(np.sum(c_prob * positions**2))

print(f'Quantum walk σ  = {q_std:.2f}  (≈ 0.71 × steps)')
print(f'Classical walk σ = {c_std:.2f} (≈ √steps)')
print(f'Speedup ratio: {q_std/c_std:.2f}x')

## 3. Decoherence: Quantum → Classical Transition
Increasing γ (gamma) introduces environmental noise, destroying quantum coherence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0d0d1a')
gammas = [0.0, 0.3, 0.9]
colors = ['#00f5ff', '#7b61ff', '#ff4444']
labels = ['Fully Quantum (γ=0)', 'Partial Decoherence (γ=0.3)', 'Near-Classical (γ=0.9)']

for ax, gamma, col, label in zip(axes, gammas, colors, labels):
    prob = quantum_walk_1d(100, hadamard_coin(), decoherence=gamma)
    ax.set_facecolor('#0d0d1a')
    ax.fill_between(positions, prob, color=col, alpha=0.6)
    ax.plot(positions, prob, color=col, lw=1.5)
    ax.set_title(label, color='white', fontsize=10)
    ax.tick_params(colors='#aaaaaa')
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')

plt.suptitle('Decoherence Erasing Quantum Walk', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/decoherence_panels.png', dpi=130, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

## 4. QFT Connection: Entropy of the Walk
Von Neumann entropy measures quantum information. As decoherence increases, entropy of the walk changes — mirroring thermalization in QFT.

In [ ]:
def shannon_entropy(prob):
    p = prob[prob > 0]
    return -np.sum(p * np.log2(p))

gammas = np.linspace(0, 1, 20)
entropies = [shannon_entropy(quantum_walk_1d(80, hadamard_coin(), d)) for d in gammas]

plt.figure(figsize=(8, 4), facecolor='#0d0d1a')
ax = plt.gca()
ax.set_facecolor('#0d0d1a')
ax.plot(gammas, entropies, color='#00f5ff', lw=2.5, marker='o', markersize=5)
ax.set_xlabel('Decoherence γ', color='#aaaaaa')
ax.set_ylabel('Shannon Entropy (bits)', color='#aaaaaa')
ax.set_title('Entropy vs Decoherence — Quantum to Classical Transition', color='white')
ax.tick_params(colors='#aaaaaa')
for spine in ax.spines.values(): spine.set_edgecolor('#333355')
plt.tight_layout()
plt.show()

## 5. Grover Coin vs Hadamard Coin
Different coin operators lead to different walk patterns — analogous to choosing different interaction Hamiltonians in QFT.

In [ ]:
h_prob = quantum_walk_1d(100, hadamard_coin(), decoherence=0.0)
g_prob = quantum_walk_1d(100, grover_coin(), decoherence=0.0)

plt.figure(figsize=(10, 4), facecolor='#0d0d1a')
ax = plt.gca()
ax.set_facecolor('#0d0d1a')
ax.plot(positions, h_prob, color='#00f5ff', lw=1.8, label='Hadamard Coin')
ax.plot(positions, g_prob, color='#ff6b9d', lw=1.8, label='Grover Coin', linestyle='--')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
ax.set_title('Coin Operator Comparison', color='white')
ax.tick_params(colors='#aaaaaa')
for spine in ax.spines.values(): spine.set_edgecolor('#333355')
plt.tight_layout()
plt.show()